<p style="text-align:center">
    <a href="https://skills.network/?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMDS0321ENSkillsNetwork26802033-2022-01-01" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


# **Hands-on Lab: Interactive Visual Analytics with Folium**


Estimated time needed: **40** minutes


The launch success rate may depend on many factors such as payload mass, orbit type, and so on. It may also depend on the location and proximities of a launch site, i.e., the initial position of rocket trajectories. Finding an optimal location for building a launch site certainly involves many factors and hopefully we could discover some of the factors by analyzing the existing launch site locations.


In the previous exploratory data analysis labs, you have visualized the SpaceX launch dataset using `matplotlib` and `seaborn` and discovered some preliminary correlations between the launch site and success rates. In this lab, you will be performing more interactive visual analytics using `Folium`.


## Objectives


This lab contains the following tasks:

*   **TASK 1:** Mark all launch sites on a map
*   **TASK 2:** Mark the success/failed launches for each site on the map
*   **TASK 3:** Calculate the distances between a launch site to its proximities

After completed the above tasks, you should be able to find some geographical patterns about launch sites.


Let's first import required Python packages for this lab:


In [1]:
import piplite
await piplite.install(['folium'])
await piplite.install(['pandas'])

In [2]:
import folium
import pandas as pd

In [3]:
# Import folium MarkerCluster plugin
from folium.plugins import MarkerCluster
# Import folium MousePosition plugin
from folium.plugins import MousePosition
# Import folium DivIcon plugin
from folium.features import DivIcon

If you need to refresh your memory about folium, you may download and refer to this previous folium lab:


[Generating Maps with Python](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/labs/v4/DV0101EN-Exercise-Generating-Maps-in-Python.ipynb)


In [ ]:
## Task 1: Mark all launch sites on a map


First, let's try to add each site's location on a map using site's latitude and longitude coordinates


The following dataset with the name `spacex_launch_geo.csv` is an augmented dataset with latitude and longitude added for each site.


In [4]:
# Download and read the `spacex_launch_geo.csv`
from js import fetch
import io

URL = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv'
resp = await fetch(URL)
spacex_csv_file = io.BytesIO((await resp.arrayBuffer()).to_py())
spacex_df=pd.read_csv(spacex_csv_file)

Now, you can take a look at what are the coordinates for each site.


In [5]:
# Select relevant sub-columns: `Launch Site`, `Lat(Latitude)`, `Long(Longitude)`, `class`
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]
launch_sites_df

,Launch Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,CCAFS SLC-40,28.563197,-80.576820
2,KSC LC-39A,28.573255,-80.646895
3,VAFB SLC-4E,34.632834,-120.610745


Above coordinates are just plain numbers that can not give you any intuitive insights about where are those launch sites. If you are very good at geography, you can interpret those numbers directly in your mind. If not, that's fine too. Let's visualize those locations by pinning them on a map.


We first need to create a folium `Map` object, with an initial center location to be NASA Johnson Space Center at Houston, Texas.


In [6]:
# Start location is NASA Johnson Space Center
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=10)

We could use `folium.Circle` to add a highlighted circle area with a text label on a specific coordinate. For example,


In [7]:
# Create a blue circle at NASA Johnson Space Center's coordinate with a popup label showing its name
circle = folium.Circle(nasa_coordinate, radius=1000, color='#d35400', fill=True).add_child(folium.Popup('NASA Johnson Space Center'))
# Create a blue circle at NASA Johnson Space Center's coordinate with a icon showing its name
marker = folium.map.Marker(
    nasa_coordinate,
    # Create an icon as a text label
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'NASA JSC',
        )
    )
site_map.add_child(circle)
site_map.add_child(marker)

and you should find a small yellow circle near the city of Houston and you can zoom-in to see a larger circle.


Now, let's add a circle for each launch site in data frame `launch_sites`


*TODO:*  Create and add `folium.Circle` and `folium.Marker` for each launch site on the site map


An example of folium.Circle:


`folium.Circle(coordinate, radius=1000, color='#000000', fill=True).add_child(folium.Popup(...))`


An example of folium.Marker:


`folium.map.Marker(coordinate, icon=DivIcon(icon_size=(20,20),icon_anchor=(0,0), html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'label', ))`


In [ ]:
# TASK 1: crear un mapa y marcar cada sitio de lanzamiento.
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)

for _, sitio in launch_sites_df.iterrows():
    coordenadas = [sitio['Lat'], sitio['Long']]
    nombre = sitio['Launch Site']

    # Circulo naranja: resalta la ubicacion de la base.
    folium.Circle(
        location=coordenadas,
        radius=1000,
        color='#d35400',
        fill=True,
        fill_opacity=0.25,
        popup=nombre
    ).add_to(site_map)

    # Texto con el nombre de la base al lado del circulo.
    folium.Marker(
        location=coordenadas,
        icon=DivIcon(
            icon_size=(200, 30),
            icon_anchor=(0, 0),
            html=f'<div style="font-size:12px; color:#d35400; white-space:nowrap;"><b>{nombre}</b></div>'
        )
    ).add_to(site_map)

site_map

The generated map with marked launch sites should look similar to the following:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_markers.png">
</center>


Now, you can explore the map by zoom-in/out the marked areas
, and try to answer the following questions:

*   Are all launch sites in proximity to the Equator line?
*   Are all launch sites in very close proximity to the coast?

Also please try to explain your findings.


### Respuestas sugeridas - TASK 1

1. **Ecuador:** no estan inmediatamente sobre el ecuador. Las bases de Florida se ubican alrededor de 28 grados norte y la de California alrededor de 35 grados norte.
2. **Costa:** si, los sitios del conjunto se encuentran cerca de la costa. Esto permite planificar trayectorias que pasen sobre el oceano.


In [ ]:
# Task 2: Mark the success/failed launches for each site on the map


Next, let's try to enhance the map by adding the launch outcomes for each site, and see which sites have high success rates.
Recall that data frame spacex_df has detailed launch records, and the `class` column indicates if this launch was successful or not


In [ ]:
spacex_df.tail(10)

Next, let's create markers for all launch records.
If a launch was successful `(class=1)`, then we use a green marker and if a launch was failed, we use a red marker `(class=0)`


Note that a launch only happens in one of the four launch sites, which means many launch records will have the exact same coordinate. Marker clusters can be a good way to simplify a map containing many markers having the same coordinate.


Let's first create a `MarkerCluster` object


In [ ]:
marker_cluster = MarkerCluster()


*TODO:* Create a new column in `spacex_df` dataframe called `marker_color` to store the marker colors based on the `class` value


In [ ]:
# TASK 2: verde si class=1 (exito), rojo si class=0 (fallo).
spacex_df['marker_color'] = spacex_df['class'].map({1: 'green', 0: 'red'})
spacex_df[['Launch Site', 'class', 'marker_color']].head()

*TODO:* For each launch result in `spacex_df` data frame, add a `folium.Marker` to `marker_cluster`


In [ ]:
# TASK 2: agregar el grupo de marcadores al mapa.
site_map.add_child(marker_cluster)

# Cada fila de spacex_df representa un lanzamiento.
for _, record in spacex_df.iterrows():
    coordenadas = [record['Lat'], record['Long']]
    resultado = 'Exito' if record['class'] == 1 else 'Fallo'

    marcador = folium.Marker(
        location=coordenadas,
        popup=f"{record['Launch Site']} - {resultado}",
        icon=folium.Icon(color=record['marker_color'])
    )
    marker_cluster.add_child(marcador)

site_map

Your updated map may look like the following screenshots:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_cluster.png">
</center>


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_cluster_zoomed.png">
</center>


From the color-labeled markers in marker clusters, you should be able to easily identify which launch sites have relatively high success rates.


In [ ]:
# TASK 3: Calculate the distances between a launch site to its proximities


Next, we need to explore and analyze the proximities of launch sites.


Let's first add a `MousePosition` on the map to get coordinate for a mouse over a point on the map. As such, while you are exploring the map, you can easily find the coordinates of any points of interests (such as railway)


In [ ]:
# Add Mouse Position to get the coordinate (Lat, Long) for a mouse over on the map
formatter = "function(num) {return L.Util.formatNum(num, 5);};"
mouse_position = MousePosition(
    position='topright',
    separator=' Long: ',
    empty_string='NaN',
    lng_first=False,
    num_digits=20,
    prefix='Lat:',
    lat_formatter=formatter,
    lng_formatter=formatter,
)

site_map.add_child(mouse_position)
site_map

Now zoom in to a launch site and explore its proximity to see if you can easily find any railway, highway, coastline, etc. Move your mouse to these points and mark down their coordinates (shown on the top-left) in order to the distance to the launch site.


Now zoom in to a launch site and explore its proximity to see if you can easily find any railway, highway, coastline, etc. Move your mouse to these points and mark down their coordinates (shown on the top-left) in order to the distance to the launch site.


In [ ]:
from math import sin, cos, sqrt, atan2, radians

def calculate_distance(lat1, lon1, lat2, lon2):
    # approximate radius of earth in km
    R = 6373.0

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    distance = R * c
    return distance

*TODO:* Mark down a point on the closest coastline using MousePosition and calculate the distance between the coastline point and the launch site.


In [ ]:
# TASK 3: seleccionar una base y un punto aproximado de la costa.
# Este punto de costa es el ejemplo dado por el laboratorio; revisalo en el mapa.
sitio_elegido = launch_sites_df.loc[
    launch_sites_df['Launch Site'] == 'CCAFS SLC-40'
].iloc[0]

coordenadas_lanzamiento = [sitio_elegido['Lat'], sitio_elegido['Long']]
coordenadas_costa = [28.56367, -80.57163]

distance_coastline = calculate_distance(
    coordenadas_lanzamiento[0], coordenadas_lanzamiento[1],
    coordenadas_costa[0], coordenadas_costa[1]
)
print(f"Distancia aproximada a la costa: {distance_coastline:.2f} km")

In [ ]:
# TASK 3: marcar el punto de la costa y mostrar la distancia.
folium.CircleMarker(
    location=coordenadas_costa,
    radius=5,
    color='blue',
    fill=True,
    popup='Punto de costa seleccionado'
).add_to(site_map)

distance_marker = folium.Marker(
    location=coordenadas_costa,
    icon=DivIcon(
        icon_size=(180, 30),
        icon_anchor=(0, 0),
        html=f'<div style="font-size:12px; color:blue; white-space:nowrap;"><b>Costa: {distance_coastline:.2f} km</b></div>'
    )
)
site_map.add_child(distance_marker)
site_map

*TODO:* Draw a `PolyLine` between a launch site to the selected coastline point


In [ ]:
# TASK 3: dibujar la linea recta desde el lanzamiento hasta la costa.
lines = folium.PolyLine(
    locations=[coordenadas_lanzamiento, coordenadas_costa],
    weight=3,
    color='blue',
    tooltip=f'Distancia a la costa: {distance_coastline:.2f} km'
)
site_map.add_child(lines)
site_map

Your updated map with distance line should look like the following screenshot:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_distance.png">
</center>


*TODO:* Similarly, you can draw a line betwee a launch site to its closest city, railway, highway, etc. You need to use `MousePosition` to find the their coordinates on the map first


A railway map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/railway.png">
</center>


A highway map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/highway.png">
</center>


A city map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/city.png">
</center>


In [ ]:
# TASK 3: otros puntos de referencia para practicar distancias.
# Son puntos ilustrativos, no necesariamente los mas cercanos.
# El tramo ferroviario de NASA mencionado es HISTORICO (no se supone activo).
puntos_referencia = {
    'Ciudad: Cape Canaveral': [28.393226, -80.604791],
    'Carretera: SR-528': [28.405, -80.645],
    'Ferrocarril historico NASA': [28.561522, -80.654170]
}

for nombre, destino in puntos_referencia.items():
    distancia = calculate_distance(
        coordenadas_lanzamiento[0], coordenadas_lanzamiento[1],
        destino[0], destino[1]
    )

    folium.CircleMarker(
        location=destino,
        radius=5,
        color='purple',
        fill=True,
        popup=f'{nombre}: {distancia:.2f} km'
    ).add_to(site_map)

    folium.PolyLine(
        locations=[coordenadas_lanzamiento, destino],
        weight=2,
        color='purple',
        dash_array='5, 6',
        tooltip=f'{nombre}: {distancia:.2f} km'
    ).add_to(site_map)

    print(f'{nombre}: {distancia:.2f} km')

site_map

After you plot distance lines to the proximities, you can answer the following questions easily:

*   Are launch sites in close proximity to railways?
*   Are launch sites in close proximity to highways?
*   Are launch sites in close proximity to coastline?
*   Do launch sites keep certain distance away from cities?

Also please try to explain your findings.


### Respuestas sugeridas - TASK 3

- **Ferrocarril:** depende de la instalacion. El punto de referencia de la red ferroviaria NASA usado aqui pertenece a un trazado historico; no indica por si solo si actualmente hay servicio ferroviario.
- **Carreteras:** hay carreteras en la region, pero la cercania debe evaluarse con las coordenadas de la carretera concreta y de cada base.
- **Costa:** las bases del conjunto estan cerca de zonas costeras. En el codigo se calcula una distancia aproximada a un punto seleccionado de la costa.
- **Ciudades:** el punto del centro de Cape Canaveral queda a cierta distancia de SLC-40; esto no demuestra la distancia minima a todas las zonas pobladas.

**Importante:** `calculate_distance()` calcula distancias en linea recta sobre la superficie terrestre (formula de Haversine); no mide rutas por carretera ni demuestra que el punto elegido sea realmente el mas cercano.


# Next Steps:

Now you have discovered many interesting insights related to the launch sites' location using folium, in a very interactive way. Next, you will need to build a dashboard using Ploty Dash on detailed launch records.


## Authors


[Pratiksha Verma](https://www.linkedin.com/in/pratiksha-verma-6487561b1/)


<!--## Change Log--!>


<!--| Date (YYYY-MM-DD) | Version | Changed By      | Change Description      |
| ----------------- | ------- | -------------   | ----------------------- |
| 2022-11-09        | 1.0     | Pratiksha Verma | Converted initial version to Jupyterlite|--!>


### <h3 align="center"> IBM Corporation 2022. All rights reserved. <h3/>
